# 🇦🇷 Tablero Macroeconómico Argentina — ETL (v3)

**Fuentes:** INDEC, Secretaría de Hacienda y Secretaría de Trabajo (API de Series de Tiempo de datos.gob.ar), BCRA (API Estadísticas Monetarias v4.0) y ArgentinaDatos (dólares MEP, CCL y blue).

**Novedades v3:** riesgo país (EMBI, diario y mensual, con semáforo), alerta automática de fuentes atrasadas en la hoja *cobertura*, y guardado compatible con ejecución automática (service account) además de Colab.

**Novedades v2:** dólar de convertibilidad (base monetaria / reservas netas), dólares de mercado y brecha, datos por provincia para el mapa (empleo privado registrado y exportaciones por rubro) y hoja de cobertura de datos.

**Salida:** planilla *Macro AR - Datos limpios*. **Para actualizar:** Entorno de ejecución → *Ejecutar todas* y después *Actualizar* en Power BI.

In [7]:
# =============================================================
# 1) CONFIGURACIÓN
# =============================================================
import requests, time, datetime as dt
import pandas as pd, numpy as np

SPREADSHEET_ID = "1JkyIFA9AQt4_txuVIHwI9zfNR99iUHNzsujTBAlT_m8"   # "Macro AR - Datos limpios" en tu Drive
DESDE = "2016-01-01"            # arranque de las series (IPC base dic-2016)
DIAS_DIARIO = 2700              # días de historia diaria (≈7 años) para el bloque cambiario y dólar

API_SERIES = "https://apis.datos.gob.ar/series/api/series/"
API_BCRA   = "https://api.bcra.gob.ar/estadisticas/v4.0/monetarias"
API_DOLAR  = "https://api.argentinadatos.com/v1/cotizaciones/dolares"   # MEP, CCL y blue (cotización de venta)
API_RIESGO = "https://api.argentinadatos.com/v1/finanzas/indices/riesgo-pais"   # EMBI Argentina (puntos básicos)
CSV_SSPM   = "https://infra.datos.gob.ar/catalog/sspm/dataset"
DOLARES_MERCADO = {"bolsa": ("dolar_mep", "Dólar MEP (bolsa)"),
                   "contadoconliqui": ("dolar_ccl", "Dólar contado con liquidación"),
                   "blue": ("dolar_blue", "Dólar blue (informal)")}

# polaridad: +1 = es bueno que suba | -1 = es bueno que baje | 0 = neutro
# tipo: indice_precios | indice_real | nominal_pesos | saldo_pesos | flujo_usd | saldo_usd | stock_usd | tasa_pct | fraccion | tipo_cambio
# (id_api, id_indicador, nombre, bloque, unidad, fuente, tipo, polaridad, orden)
SERIES_DATOS = [
    ("148.3_INIVELNAL_DICI_M_26", "ipc_general",      "IPC Nivel general",              "Precios",               "Índice dic-16=100",  "INDEC", "indice_precios", -1, 1),
    ("148.3_INUCLEONAL_DICI_M_19","ipc_nucleo",       "IPC Núcleo",                     "Precios",               "Índice dic-16=100",  "INDEC", "indice_precios", -1, 2),
    ("148.3_IREGULANAL_DICI_M_22","ipc_regulados",    "IPC Regulados",                  "Precios",               "Índice dic-16=100",  "INDEC", "indice_precios", -1, 3),
    ("148.3_IESTACINAL_DICI_M_25","ipc_estacionales", "IPC Estacionales",               "Precios",               "Índice dic-16=100",  "INDEC", "indice_precios", -1, 4),
    ("143.3_NO_PR_2004_A_31",     "emae_desest",      "EMAE desestacionalizado",        "Actividad y empleo",    "Índice 2004=100",    "INDEC", "indice_real",    +1, 10),
    ("143.3_NO_PR_2004_A_21",     "emae_original",    "EMAE original",                  "Actividad y empleo",    "Índice 2004=100",    "INDEC", "indice_real",    +1, 11),
    ("11.3_VMASD_2004_M_23",      "emae_industria",   "EMAE Industria manufacturera",   "Actividad y empleo",    "Índice 2004=100",    "INDEC", "indice_real",    +1, 12),
    ("11.3_VMATC_2004_M_12",      "emae_construccion","EMAE Construcción",              "Actividad y empleo",    "Índice 2004=100",    "INDEC", "indice_real",    +1, 13),
    ("11.3_ISOM_2004_M_39",       "emae_agro",        "EMAE Agro",                      "Actividad y empleo",    "Índice 2004=100",    "INDEC", "indice_real",    +1, 14),
    ("11.3_VIPAA_2004_M_5", "emae_pesca", "EMAE Pesca", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 100),
    ("11.3_ISD_2004_M_26", "emae_minas", "EMAE Minas y canteras", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 101),
    ("11.3_ITC_2004_M_21", "emae_electricidad", "EMAE Electricidad, gas y agua", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 102),
    ("11.3_AGCS_2004_M_41", "emae_comercio", "EMAE Comercio", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 103),
    ("11.3_P_2004_M_20", "emae_hoteles", "EMAE Hoteles y restaurantes", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 104),
    ("11.3_EMC_2004_M_25", "emae_transporte", "EMAE Transporte y comunicaciones", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 105),
    ("11.3_IM_2004_M_25", "emae_financiero", "EMAE Intermediación financiera", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 106),
    ("11.3_SEGA_2004_M_48", "emae_inmobiliario", "EMAE Inmobiliarias y empresariales", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 107),
    ("11.3_C_2004_M_60", "emae_admin_publica", "EMAE Administración pública", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 108),
    ("11.3_CMMR_2004_M_10", "emae_ensenianza", "EMAE Enseñanza", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 109),
    ("11.3_HR_2004_M_24", "emae_salud", "EMAE Salud", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 110),
    ("11.3_TAC_2004_M_60", "emae_serv_comunitarios", "EMAE Servicios comunitarios y personales", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 111),
    ("11.3_IF_2004_M_25", "emae_impuestos", "EMAE Impuestos netos de subsidios", "Actividad y empleo", "Índice 2004=100", "INDEC", "indice_real", +1, 112),
    ("4.2_OGP_2004_T_17",         "pib_real",         "PIB real (trimestral)",          "Actividad y empleo",    "Millones de $ de 2004","INDEC","indice_real",    +1, 15),
    ("42.3_EPH_PUNTUATAL_0_M_30", "desocupacion",     "Tasa de desocupación",           "Actividad y empleo",    "%",                  "INDEC", "fraccion",       -1, 16),
    ("42.3_EPH_PUNTUATAL_0_M_24", "empleo",           "Tasa de empleo",                 "Actividad y empleo",    "%",                  "INDEC", "fraccion",       +1, 17),
    ("149.1_TL_INDIIOS_OCTU_0_21","salarios",         "Índice de salarios (total)",     "Actividad y empleo",    "Índice oct-16=100",  "INDEC", "nominal_pesos",  +1, 18),
    ("149.1_SOR_PRIADO_OCTU_0_25","salarios_priv_reg","Salarios privados registrados",  "Actividad y empleo",    "Índice oct-16=100",  "INDEC", "nominal_pesos",  +1, 19),
    ("149.1_SOR_PUBICO_OCTU_0_14","salarios_publico", "Salarios sector público",        "Actividad y empleo",    "Índice oct-16=100",  "INDEC", "nominal_pesos",  +1, 20),
    ("158.1_REPTE_0_0_5",         "ripte",            "RIPTE",                          "Actividad y empleo",    "$ corrientes",       "Secretaría de Trabajo", "nominal_pesos", +1, 21),
    ("74.3_IET_0_M_16",           "exportaciones",    "Exportaciones",                  "Externo y fiscal",      "Millones de USD",    "INDEC", "flujo_usd",      +1, 40),
    ("74.3_IIT_0_M_25",           "importaciones",    "Importaciones",                  "Externo y fiscal",      "Millones de USD",    "INDEC", "flujo_usd",       0, 41),
    ("74.3_ISC_0_M_19",           "saldo_comercial",  "Saldo comercial",                "Externo y fiscal",      "Millones de USD",    "INDEC", "saldo_usd",      +1, 42),
    ("452.3_RESULTADO_RIO_0_M_18_54","resultado_primario","Resultado primario (SPN)",   "Externo y fiscal",      "Millones de $",      "Secretaría de Hacienda", "saldo_pesos",   +1, 43),
    ("452.3_RESULTADO_ERO_0_M_20_25","resultado_financiero","Resultado financiero (SPN)","Externo y fiscal",     "Millones de $",      "Secretaría de Hacienda", "saldo_pesos",   +1, 44),
    ("452.3_INTERESES_TOS_0_M_15_62","intereses_deuda","Intereses netos de la deuda",   "Externo y fiscal",      "Millones de $",      "Secretaría de Hacienda", "nominal_pesos", -1, 45),
    ("172.3_TL_RECAION_M_0_0_17", "recaudacion",      "Recaudación tributaria",         "Externo y fiscal",      "Millones de $",      "Secretaría de Hacienda", "nominal_pesos", +1, 46),
]

# (idVariable, id_indicador, nombre, bloque, unidad, tipo, polaridad, orden, agregacion_mensual)
SERIES_BCRA = [
    (1,    "reservas",         "Reservas internacionales brutas", "Cambiario y monetario", "Millones de USD",  "stock_usd",   +1, 30, "last"),
    (5,    "tc_mayorista",     "Tipo de cambio mayorista (A3500)","Cambiario y monetario", "$ por USD",        "tipo_cambio",  0, 31, "mean"),
    (4,    "tc_minorista",     "Tipo de cambio minorista",        "Cambiario y monetario", "$ por USD",        "tipo_cambio",  0, 32, "mean"),
    (1187, "banda_inferior",   "Banda cambiaria - piso",          "Cambiario y monetario", "$ por USD",        "tipo_cambio",  0, 33, "last"),
    (1188, "banda_superior",   "Banda cambiaria - techo",         "Cambiario y monetario", "$ por USD",        "tipo_cambio",  0, 34, "last"),
    (44,   "tamar",            "Tasa TAMAR bancos privados",      "Cambiario y monetario", "% TNA",            "tasa_pct",     0, 35, "mean"),
    (7,    "badlar",           "Tasa BADLAR bancos privados",     "Cambiario y monetario", "% TNA",            "tasa_pct",     0, 36, "mean"),
    (15,   "base_monetaria",   "Base monetaria",                  "Cambiario y monetario", "Millones de $",    "nominal_pesos", 0, 37, "last"),
    (26,   "prestamos_privados","Préstamos al sector privado",    "Cambiario y monetario", "Millones de $",    "nominal_pesos",+1, 38, "last"),
    (24,   "plazos_fijos",     "Depósitos a plazo",               "Cambiario y monetario", "Millones de $",    "nominal_pesos",+1, 39, "last"),
    (25,   "m2_privado_ia",    "M2 privado (var. % i.a.)",        "Cambiario y monetario", "% i.a.",           "tasa_pct",     0, 39.5, "last"),
    (29,   "inflacion_esperada","Inflación esperada 12 meses (REM)","Precios",             "% i.a.",           "tasa_pct",    -1, 5, "last"),
]

sesion = requests.Session()
sesion.headers.update({"User-Agent": "tablero-macro-ar/1.0"})

def get_json(url, params=None, intentos=4):
    for i in range(intentos):
        try:
            r = sesion.get(url, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if i == intentos - 1:
                raise
            time.sleep(2 * (i + 1))

print("Configuración lista:", len(SERIES_DATOS), "series de datos.gob.ar y", len(SERIES_BCRA), "del BCRA")


Configuración lista: 23 series de datos.gob.ar y 12 del BCRA


## Descarga

In [8]:
# =============================================================
# 2) DESCARGA: API Series de Tiempo (INDEC / Hacienda / Trabajo) + API BCRA
# =============================================================
crudo = []          # filas: fecha, id_indicador, valor, frecuencia
diario = []         # filas diarias del BCRA (últimos DIAS_DIARIO días)
errores = []

# --- datos.gob.ar (una serie por llamada para respetar su frecuencia) ---
for (sid, ind, *_rest) in SERIES_DATOS:
    try:
        j = get_json(API_SERIES, {"ids": sid, "start_date": DESDE, "limit": 5000, "format": "json", "metadata": "simple"})
        freq = j["meta"][0].get("frequency", "month")
        for fecha, valor in j["data"]:
            if valor is not None:
                crudo.append((fecha, ind, float(valor), freq))
        print(f"✔ {ind:22s} {len(j['data']):5d} obs  último: {j['data'][-1][0] if j['data'] else '-'}")
    except Exception as e:
        errores.append((ind, str(e)[:150]))
        print(f"✘ {ind}: {e}")

# --- BCRA v4.0 (diarias / mensuales, paginadas) ---
def bajar_bcra(id_var, desde=DESDE):
    filas, offset, LIM = [], 0, 3000
    while True:
        j = get_json(f"{API_BCRA}/{id_var}", {"desde": desde, "limit": LIM, "offset": offset})
        res = j.get("results", [])
        det = res[0]["detalle"] if res else []
        filas += [(d["fecha"], d["valor"]) for d in det]
        total = j.get("metadata", {}).get("resultset", {}).get("count", 0)
        offset += LIM
        if offset >= total or not det:
            break
    return pd.DataFrame(filas, columns=["fecha", "valor"])

hoy = pd.Timestamp.today().normalize()
diarios_full = {}   # series diarias completas (para cálculos como la convertibilidad)
for (idv, ind, nombre, bloque, unidad, tipo, pol, orden, agg) in SERIES_BCRA:
    try:
        df = bajar_bcra(idv)
        df["fecha"] = pd.to_datetime(df["fecha"])
        df = df.dropna().sort_values("fecha")
        df = df[df["fecha"] <= hoy]          # descarta valores programados a futuro (ej. bandas)
        diarios_full[ind] = df.set_index("fecha")["valor"]
        # mensual: promedio o último dato del mes
        m = df.set_index("fecha")["valor"].resample("MS")
        m = (m.mean() if agg == "mean" else m.last()).dropna()
        crudo += [(f.strftime("%Y-%m-%d"), ind, float(v), "month") for f, v in m.items()]
        # diario reciente
        rec = df[df["fecha"] >= hoy - pd.Timedelta(days=DIAS_DIARIO)]
        diario += [(f.strftime("%Y-%m-%d"), ind, float(v)) for f, v in zip(rec["fecha"], rec["valor"])]
        print(f"✔ {ind:22s} {len(df):5d} obs  último: {df['fecha'].max().date()}")
    except Exception as e:
        errores.append((ind, str(e)[:150]))
        print(f"✘ {ind}: {e}")

# --- Dólares de mercado (MEP, CCL, blue) - ArgentinaDatos ---
for casa, (ind, nombre) in DOLARES_MERCADO.items():
    try:
        j = get_json(f"{API_DOLAR}/{casa}")
        df = pd.DataFrame(j)[["fecha", "venta"]].rename(columns={"venta": "valor"})
        df["fecha"] = pd.to_datetime(df["fecha"])
        df = df[(df["fecha"] >= DESDE) & (df["fecha"] <= hoy)].dropna().drop_duplicates("fecha").sort_values("fecha")
        s_ = df.set_index("fecha")["valor"]
        diarios_full[ind] = s_
        crudo += [(f.strftime("%Y-%m-%d"), ind, float(v), "month") for f, v in s_.resample("MS").mean().dropna().items()]
        rec = s_[s_.index >= hoy - pd.Timedelta(days=DIAS_DIARIO)]
        diario += [(f.strftime("%Y-%m-%d"), ind, float(v)) for f, v in rec.items()]
        print(f"✔ {ind:22s} {len(s_):5d} obs  último: {s_.index.max().date()}")
    except Exception as e:
        errores.append((ind, str(e)[:150])); print(f"✘ {ind}: {e}")

# --- Riesgo país (EMBI) - ArgentinaDatos ---
try:
    j = get_json(API_RIESGO)
    rp = pd.DataFrame(j)
    rp["fecha"] = pd.to_datetime(rp["fecha"])
    rp = rp[(rp["fecha"] >= DESDE) & (rp["fecha"] <= hoy)].dropna().drop_duplicates("fecha").sort_values("fecha")
    s_ = rp.set_index("fecha")["valor"].astype(float)
    diarios_full["riesgo_pais"] = s_
    crudo += [(f.strftime("%Y-%m-%d"), "riesgo_pais", float(v), "month") for f, v in s_.resample("MS").mean().dropna().items()]
    diario += [(f.strftime("%Y-%m-%d"), "riesgo_pais", float(v)) for f, v in s_[s_.index >= hoy - pd.Timedelta(days=DIAS_DIARIO)].items()]
    print(f"✔ {'riesgo_pais':22s} {len(s_):5d} obs  último: {s_.index.max().date()}")
except Exception as e:
    errores.append(("riesgo_pais", str(e)[:150])); print(f"✘ riesgo_pais: {e}")

# --- Dólar de convertibilidad = Base monetaria / Reservas brutas, y brecha MEP vs. oficial ---
def agregar_diaria(ind, serie):
    serie = serie.dropna()
    crudo.extend((f.strftime("%Y-%m-%d"), ind, float(v), "month") for f, v in serie.resample("MS").mean().dropna().items())
    rec = serie[serie.index >= hoy - pd.Timedelta(days=DIAS_DIARIO)]
    diario.extend((f.strftime("%Y-%m-%d"), ind, float(v)) for f, v in rec.items())

if {"base_monetaria", "reservas"} <= set(diarios_full):
    conv = (diarios_full["base_monetaria"] / diarios_full["reservas"]).dropna()   # millones de $ / millones de USD = $ por USD
    agregar_diaria("dolar_convertibilidad", conv)
    print(f"✔ dolar_convertibilidad  último: {conv.index.max().date()}  = ${conv.iloc[-1]:,.0f}")
if {"dolar_mep", "tc_mayorista"} <= set(diarios_full):
    agregar_diaria("brecha_mep", (diarios_full["dolar_mep"] / diarios_full["tc_mayorista"] - 1).dropna() * 100)

DIARIAS_TABLERO = {"tc_mayorista", "tc_minorista", "banda_inferior", "banda_superior", "reservas", "base_monetaria",
                   "dolar_mep", "dolar_ccl", "dolar_blue", "dolar_convertibilidad", "brecha_mep", "riesgo_pais"}
diario = [r for r in diario if r[1] in DIARIAS_TABLERO]

base = pd.DataFrame(crudo, columns=["fecha", "id_indicador", "valor", "frecuencia"])
base["fecha"] = pd.to_datetime(base["fecha"])
base_diaria = pd.DataFrame(diario, columns=["fecha", "id_indicador", "valor"])
print("\nTotal:", len(base), "observaciones mensuales/trimestrales |", len(base_diaria), "diarias")
if errores:
    print("⚠ Series con error:", errores)


✔ ipc_general              119 obs  último: 2026-10-01
✔ ipc_nucleo               119 obs  último: 2026-10-01
✔ ipc_regulados            119 obs  último: 2026-10-01
✔ ipc_estacionales         119 obs  último: 2026-10-01
✔ emae_desest              128 obs  último: 2026-08-01
✔ emae_original            128 obs  último: 2026-08-01
✔ emae_industria           128 obs  último: 2026-08-01
✔ emae_construccion        128 obs  último: 2026-08-01
✔ emae_agro                128 obs  último: 2026-08-01
✔ pib_real                  42 obs  último: 2026-04-01
✔ desocupacion              41 obs  último: 2026-04-01
✔ empleo                    41 obs  último: 2026-04-01
✔ salarios                 119 obs  último: 2026-08-01
✔ salarios_priv_reg        128 obs  último: 2026-08-01
✔ salarios_publico         128 obs  último: 2026-08-01
✔ ripte                    128 obs  último: 2026-08-01
✔ exportaciones            129 obs  último: 2026-09-01
✔ importaciones            129 obs  último: 2026-09-01
✔ saldo_co

In [ ]:
# =============================================================
# 2c) RESERVAS NETAS y DÓLAR DE CONVERTIBILIDAD = base monetaria / reservas NETAS
# =============================================================
# Netas = brutas - encajes de los depósitos en USD - swap con China - otros pasivos en USD a menos de
# 12 meses (repos, Sedesa, organismos internacionales, BOPREAL). Criterio tipo FMI / consultoras: ESTIMACIÓN.
# Lo que el BCRA no publica como serie diaria va fijo acá (millones de USD): revisarlo cada tanto.
PASIVOS_FIJOS_USD = {"swap_china": 18000, "repos_sedesa_organismos_bopreal": 6000}   # ref.: informes privados, nov-2026
ENCAJES_RESPALDO_USD = 16700   # encajes en USD si no aparece la serie en la API del BCRA (ref. nov-2026)
NETAS_DESDE = "2025-01-01"     # los pasivos fijos solo representan el período reciente
NETAS_MINIMO_CONV = 5000       # con netas menores a esto el cociente explota (divide por casi cero): no se calcula
import unicodedata
_nrm = lambda s: unicodedata.normalize("NFKD", str(s or "")).encode("ascii", "ignore").decode().lower()

# --- Encajes en USD: se buscan en el catálogo de la API del BCRA; si no hay serie confiable, valor fijo ---
encajes, fuente_encajes = None, f"valor fijo USD {ENCAJES_RESPALDO_USD:,} M"
try:
    cat_bcra, off = [], 0
    while True:
        j = get_json(API_BCRA, {"limit": 1000, "offset": off})
        cat_bcra += j.get("results", [])
        off += 1000
        if off >= j.get("metadata", {}).get("resultset", {}).get("count", 0):
            break
    def _es_encaje(v):
        d, u = _nrm(v.get("descripcion")), _nrm(v.get("unidadExpresion"))
        tema = "encaje" in d or "efectivo minimo" in d or ("cuenta" in d and "bcra" in d)
        usd = "dolar" in d or "moneda extranjera" in d or "usd" in u or "dolar" in u
        return tema and usd
    for v in filter(_es_encaje, cat_bcra):
        print("  candidato encajes:", v["idVariable"], "|", v.get("descripcion"), "|", v.get("unidadExpresion"))
        try:
            s_e = bajar_bcra(v["idVariable"], desde="2024-01-01")
            s_e["fecha"] = pd.to_datetime(s_e["fecha"])
            s_e = s_e.dropna().set_index("fecha")["valor"].sort_index()
            if s_e.iloc[-1] > 1e6:                      # viene en miles de USD
                s_e = s_e / 1000
            if 5000 <= s_e.iloc[-1] <= 40000:           # rango razonable en millones de USD
                encajes, fuente_encajes = s_e, f"BCRA id {v['idVariable']}: {v.get('descripcion')}"
                break
        except Exception as e:
            print("    (descartado:", str(e)[:80], ")")
except Exception as e:
    errores.append(("encajes_usd", str(e)[:150])); print("✘ catálogo BCRA:", e)
print("Encajes usados:", fuente_encajes)

# --- Cálculo diario ---
if {"base_monetaria", "reservas"} <= set(diarios_full):
    brutas = diarios_full["reservas"]
    brutas = brutas[brutas.index >= NETAS_DESDE]
    enc = (encajes.reindex(encajes.index.union(brutas.index)).ffill().reindex(brutas.index)
           if encajes is not None else pd.Series(ENCAJES_RESPALDO_USD, index=brutas.index))
    netas = (brutas - enc.fillna(ENCAJES_RESPALDO_USD) - sum(PASIVOS_FIJOS_USD.values())).dropna()
    conv_n = (diarios_full["base_monetaria"].reindex(netas.index) / netas.where(netas >= NETAS_MINIMO_CONV)).dropna()
    # reemplaza la convertibilidad calculada con brutas
    crudo[:] = [r for r in crudo if r[1] != "dolar_convertibilidad"]
    diario[:] = [r for r in diario if r[1] != "dolar_convertibilidad"]
    agregar_diaria("reservas_netas", netas)
    agregar_diaria("dolar_convertibilidad", conv_n)
    diarios_full["reservas_netas"], diarios_full["dolar_convertibilidad"] = netas, conv_n
    SERIES_BCRA.append(("calc", "reservas_netas", "Reservas internacionales netas (estimación)", "Cambiario y monetario",
                        "Millones de USD", "stock_usd", +1, 30.5, "last"))
    print(f"✔ reservas_netas          último: {netas.index.max().date()}  = USD {netas.iloc[-1]:,.0f} M "
          f"(brutas {brutas.iloc[-1]:,.0f} - encajes {enc.iloc[-1]:,.0f} - otros {sum(PASIVOS_FIJOS_USD.values()):,})")
    if len(conv_n):
        print(f"✔ dolar_convertibilidad  último: {conv_n.index.max().date()}  = ${conv_n.iloc[-1]:,.0f} (con reservas netas)")
    else:
        print(f"⚠ reservas netas por debajo de USD {NETAS_MINIMO_CONV:,} M: no se calcula la convertibilidad")

    base = pd.DataFrame(crudo, columns=["fecha", "id_indicador", "valor", "frecuencia"])
    base["fecha"] = pd.to_datetime(base["fecha"])
    base_diaria = pd.DataFrame(diario, columns=["fecha", "id_indicador", "valor"])

## Datos por provincia (mapa)

In [9]:
# =============================================================
# 2b) DATOS POR PROVINCIA (para el mapa): empleo privado registrado y exportaciones por rubro
# =============================================================
# Formas simplificadas de las provincias (SVG 300x602) a partir de la cartografía del IGN.
# (id, nombre, región, columna_asalariados, prefijo_exportaciones, path_svg, cx, cy)
PROVINCIAS = [["buenos_aires", "Buenos Aires", "Pampeana", "buenos_aires", "buenos_aires", "M173,325 172,332 170,334 171,337 172,338 172,338 172,338 174,339 172,344 166,347 163,343 157,341 157,228 181,228 193,213 199,214 201,208 211,215 212,214 213,216 216,216 216,217 219,218 224,222 228,221 228,224 226,226 227,230 226,233 227,234 229,232 231,234 235,236 240,239 243,241 244,244 246,247 242,252 243,257 246,262 249,263 251,263 251,262 251,263 252,265 252,273 246,284 241,290 240,295 236,298 226,302 208,308 189,311 180,311 176,309 172,306 170,306 169,307 172,309 172,314 174,315 172,316 175,316 177,317 176,318 174,316 173,316 174,318 176,319 175,326Z", 203, 279],
    ["caba", "CABA", "Pampeana", "caba", "ciudad_de_buenos_aires", "M227,233 227,234 226,233 226,232 226,231 227,230 227,231 227,230 227,231 227,231 228,231 228,231 228,231 228,232 229,232 229,232 229,232 229,232 229,232 228,233Z", 227, 232],
    ["catamarca", "Catamarca", "NOA", "catamarca", "catamarca", "M128,151 123,134 119,130 115,129 115,126 113,124 112,118 106,118 104,120 96,119 94,120 93,115 90,116 90,115 88,115 88,114 86,113 86,111 85,110 84,108 84,110 81,109 76,109 77,106 79,105 78,103 81,101 80,100 80,98 82,97 84,98 84,97 85,97 85,96 86,97 88,96 88,94 84,86 84,82 87,80 84,67 85,66 85,62 112,65 112,66 113,67 112,70 111,72 108,72 108,77 112,80 113,82 112,82 113,84 115,85 117,80 118,80 119,81 118,89 120,88 123,91 122,93 122,94 118,100 121,101 121,105 123,110 124,109 125,113 126,113 129,110 132,112 133,122 133,124 132,124 132,124 136,128 136,135 138,136 137,137 137,138 137,141 135,142 135,146 132,150Z", 99, 107],
    ["chaco", "Chaco", "NEA", "chaco", "chaco", "M181,113 181,91 181,71 157,71 172,48 172,43 175,45 181,47 185,52 188,53 189,56 192,59 196,61 196,62 198,62 201,66 203,71 207,74 210,77 210,80 213,80 214,83 217,83 220,84 225,90 227,92 228,93 226,94 227,95 226,96 226,96 226,97 224,98 225,99 225,101 222,102 221,104 222,109 221,114 221,113Z", 195, 79],
    ["chubut", "Chubut", "Patagonia", "chubut", "chubut", "M98,436 40,436 41,435 39,434 38,431 39,428 43,427 42,425 45,424 45,423 42,420 42,418 35,417 34,414 43,414 45,415 47,414 47,411 48,411 48,410 47,408 44,407 38,408 38,405 38,404 38,402 39,402 40,399 39,397 41,394 39,392 37,392 38,391 36,391 37,388 39,388 39,386 36,384 33,381 34,378 33,377 35,373 35,372 34,371 33,367 35,367 36,368 38,367 39,367 39,365 133,365 136,368 142,369 140,372 147,372 148,371 148,369 143,369 151,366 152,366 154,370 154,371 154,374 154,371 153,369 154,376 154,378 152,379 147,380 145,378 145,376 142,374 139,374 135,376 134,379 144,382 135,387 134,390 130,394 130,399 132,401 130,404 131,407 130,409 130,410 125,412 124,415 124,416 127,417 125,420 118,418 116,420 113,420 114,421 112,421 113,422 107,423 101,430 101,433Z", 85, 400],
    ["cordoba", "Córdoba", "Pampeana", "cordoba", "cordoba", "M157,228 157,239 133,239 132,206 134,203 136,194 135,190 131,191 131,187 130,186 124,183 124,168 128,151 132,150 135,146 135,142 144,140 149,141 149,142 151,141 152,143 156,143 156,145 165,146 166,148 169,150 168,154 170,155 174,155 175,158 179,162 175,178 173,180 175,185 174,187 178,193 178,196 178,197 180,199 180,203 164,228Z", 153, 189],
    ["corrientes", "Corrientes", "NEA", "corrientes", "corrientes", "M239,152 238,153 239,155 235,158 236,162 233,160 232,157 231,154 225,152 221,153 218,153 216,156 211,155 211,157 209,154 211,149 210,146 211,138 212,137 212,135 216,132 219,122 218,117 218,115 220,116 222,113 222,107 221,104 222,102 226,100 233,100 238,101 240,103 243,103 244,104 248,104 248,105 250,106 251,106 252,106 252,105 252,104 253,103 256,106 258,103 261,101 261,103 264,109 264,112 267,116 265,118 266,119 266,121 264,120 263,122 262,122 262,124 258,127 256,133 253,133 250,140 246,145 244,146 243,149ZM249,103 251,104 251,105 250,106 249,106 249,105 248,104 247,104Z", 237, 129],
    ["entre_rios", "Entre Ríos", "Pampeana", "entre_rios", "entre_rios", "M228,206 226,213 228,221 224,222 219,218 216,217 216,216 213,216 212,214 211,215 210,214 208,214 208,212 203,209 201,206 198,205 196,201 194,194 196,191 196,186 195,185 196,183 196,182 203,175 204,172 205,172 209,165 210,162 209,157 210,155 211,157 211,155 216,156 218,153 221,153 225,152 231,154 232,157 233,160 236,162 236,165 235,166 235,168 235,171 234,174 232,175 234,177 233,181 231,183 232,185 231,188 232,190 231,193 232,203Z", 213, 187],
    ["formosa", "Formosa", "NEA", "formosa", "formosa", "M230,84 231,89 230,89 230,91 228,93 220,84 217,83 214,83 213,80 210,80 210,77 207,74 203,71 201,66 198,62 196,62 196,61 192,59 189,56 188,53 185,52 181,47 172,43 172,14 173,15 173,16 174,17 174,18 178,25 178,25 180,26 181,28 184,30 184,31 190,34 191,38 201,42 205,41 213,47 218,52 227,57 229,59 230,58 237,62 238,65 240,67 239,69 236,73 236,74 235,76 236,77 231,81 232,81Z", 205, 54],
    ["jujuy", "Jujuy", "NOA", "jujuy", "jujuy", "M130,50 125,48 124,42 120,41 120,32 119,31 118,31 116,34 115,43 113,45 108,43 103,36 106,23 104,20 106,17 106,15 109,13 110,10 116,7 117,1 119,2 120,4 121,4 124,7 132,7 129,16 131,17 131,21 133,24 133,28 135,28 136,32 139,31 142,35 144,33 146,33 146,45 145,49 143,49 139,52 137,50 135,52 132,50Z", 120, 26],
    ["la_pampa", "La Pampa", "Pampeana", "la_pampa", "la_pampa", "M141,308 133,307 129,308 127,307 125,308 119,306 114,306 112,305 112,304 111,302 104,299 103,297 98,297 96,294 94,294 93,292 96,290 95,287 88,285 88,260 88,260 88,257 133,257 133,239 157,239 157,317 155,316 148,311Z", 123, 274],
    ["la_rioja", "La Rioja", "NOA", "la_rioja", "la_rioja", "M98,156 98,154 91,147 90,147 88,145 85,143 80,142 79,142 78,143 78,142 78,137 78,134 79,133 78,130 79,129 76,127 73,121 72,121 69,121 69,120 70,119 71,116 72,117 76,109 81,109 84,110 84,108 85,110 86,111 86,113 88,114 88,115 90,115 90,116 93,115 94,120 96,119 104,120 106,118 112,118 113,124 115,126 115,129 119,130 123,134 128,151 124,168 124,183 121,183 119,184 116,184 115,185 109,182 108,178 107,178 105,174 105,165Z", 108, 146],
    ["mendoza", "Mendoza", "Cuyo", "mendoza", "mendoza", "M88,260 88,260 88,285 86,285 85,283 82,283 81,281 77,281 77,279 74,277 70,278 68,277 67,275 67,272 65,271 62,267 61,263 59,263 58,259 59,255 58,255 59,253 58,253 58,250 58,248 57,247 58,245 55,244 56,242 58,242 59,239 60,235 59,234 64,228 63,226 65,226 67,225 65,223 66,221 65,220 66,219 65,217 67,209 65,207 64,209 63,208 62,204 64,201 62,199 61,196 62,195 63,196 64,194 66,195 68,194 68,191 69,188 72,186 73,187 75,185 79,187 79,191 83,191 85,189 85,187 91,186 94,189 99,189 99,190 101,191 104,199 103,201 104,211 108,218 111,220 110,225 109,225 109,227 110,231 112,236 114,241 111,257 88,257Z", 86, 235],
    ["misiones", "Misiones", "NEA", "misiones", "misiones", "M271,112 268,116 265,114 264,109 264,109 262,105 261,103 261,101 263,101 266,103 268,101 267,99 268,97 269,97 271,95 274,94 274,93 277,91 277,89 279,89 279,87 281,81 280,77 281,77 282,74 281,71 282,71 282,70 284,72 286,69 288,70 288,71 288,70 287,69 289,68 289,70 290,69 292,72 292,77 295,81 293,89 295,94 293,96 293,96 293,96 293,98 291,98 289,101 288,100 286,103 284,102 283,104 282,103 280,106 279,105 277,109 275,109 275,111Z", 285, 92],
    ["neuquen", "Neuquén", "Patagonia", "neuquen", "neuquen", "M38,336 39,337 40,335 38,333 38,330 40,331 40,329 41,327 40,326 40,321 42,322 43,321 42,320 44,316 43,309 48,306 51,306 52,303 50,300 49,295 50,295 47,290 48,289 47,287 48,284 46,280 48,278 48,277 46,274 48,274 47,272 48,269 49,269 49,265 50,266 51,264 54,264 54,262 56,259 57,260 58,259 59,263 61,263 62,267 65,271 67,272 67,275 68,277 70,278 74,277 77,279 77,281 81,281 82,283 85,283 86,285 88,285 88,305 90,306 92,311 88,310 84,312 78,320 74,321 71,325 65,328 64,329 64,332 63,333 63,335 61,336 61,337 56,338 54,339 50,340 48,342 49,346 46,348 37,346 37,344 36,342 38,340 38,339 37,339Z", 70, 304],
    ["rio_negro", "Río Negro", "Patagonia", "rio_negro", "rio_negro", "M133,365 39,364 39,361 37,357 38,357 38,355 37,353 37,351 38,351 37,350 38,348 37,347 42,347 45,348 48,348 49,345 48,342 50,340 54,339 56,338 61,337 61,336 63,335 63,333 64,332 64,329 65,328 71,325 74,321 78,320 84,312 88,310 92,311 90,306 88,305 88,285 95,286 96,289 93,292 94,294 96,294 98,297 103,297 104,299 111,302 112,304 112,305 113,306 119,306 125,308 126,307 129,308 133,307 143,308 148,311 155,316 157,317 157,341 160,341 163,343 166,347 161,349 152,349 141,344 136,344 138,343 137,342 134,342 136,342 133,343 132,346 132,351 135,356Z", 115, 323],
    ["salta", "Salta", "NOA", "salta", "salta", "M125,79 124,84 119,82 118,80 115,85 113,84 112,82 112,82 113,82 113,82 112,80 108,77 108,73 109,72 111,72 112,70 113,67 112,66 112,65 85,62 85,62 86,62 87,61 86,58 84,56 85,52 86,52 86,50 88,50 88,49 102,42 103,36 108,43 113,45 115,43 115,36 118,31 120,32 120,40 121,41 124,42 125,48 130,50 132,50 135,52 136,52 137,50 139,52 141,52 143,49 145,49 146,45 146,33 144,33 142,35 139,31 136,32 135,28 133,28 133,24 131,21 131,17 129,16 132,7 136,7 141,10 141,12 143,15 142,17 144,19 144,22 145,15 150,5 153,5 153,6 153,5 165,5 166,8 172,14 172,48 156,72 148,71 148,73 145,72 142,81 137,81 135,83 131,80 130,78Z", 159, 44],
    ["san_juan", "San Juan", "Cuyo", "san_juan", "san_juan", "M85,189 83,191 79,191 79,187 75,185 73,187 72,186 70,187 68,191 68,194 66,195 64,194 62,196 62,193 60,193 61,191 59,190 59,187 58,186 60,186 61,184 57,182 57,180 56,178 57,169 57,169 58,170 60,168 59,166 61,158 62,157 62,155 63,156 65,155 65,153 66,152 64,151 65,144 63,137 64,134 67,133 69,120 72,121 73,121 76,127 79,129 78,130 79,133 78,134 78,137 78,143 79,142 80,142 86,143 89,145 90,147 91,147 98,154 99,159 105,165 105,174 107,178 108,178 109,182 111,184 104,182 100,182 99,187 102,191 100,191 99,189 94,189 91,186 85,187Z", 80, 158],
    ["san_luis", "San Luis", "Cuyo", "san_luis", "san_luis", "M132,206 133,257 111,257 114,241 112,236 110,231 109,227 109,225 110,225 111,220 108,218 104,211 103,201 104,199 102,195 101,191 102,191 102,190 99,187 100,182 104,182 115,185 116,184 119,184 121,183 126,183 131,187 131,191 135,190 136,195 134,203Z", 121, 219],
    ["santa_cruz", "Santa Cruz", "Patagonia", "santa_cruz", "santa_cruz", "M77,518 76,524 71,528 75,526 79,536 76,537 73,536 71,537 69,537 78,537 82,544 87,550 86,551 86,550 80,549 75,547 71,547 64,544 35,544 36,542 31,539 31,537 29,537 31,535 31,533 30,532 32,531 30,528 30,527 32,527 32,524 30,522 31,520 28,519 27,520 25,519 20,522 17,523 17,521 19,520 15,517 14,514 13,513 14,509 13,509 14,508 13,507 15,504 14,503 15,501 13,500 15,499 14,498 15,496 14,495 15,494 19,495 21,495 19,495 19,494 22,489 24,490 28,487 27,481 29,482 30,479 30,479 32,478 30,474 29,473 29,472 28,471 29,469 28,468 29,468 31,465 31,463 35,460 35,458 37,458 37,457 36,456 37,455 36,454 36,451 40,449 39,443 39,441 37,439 39,438 40,436 98,436 97,441 100,446 104,449 109,454 121,456 124,458 124,463 123,468 120,467 116,470 122,468 122,470 124,471 121,471 120,473 121,473 122,474 119,475 116,477 115,479 114,479 113,480 110,481 98,491 97,494 94,497 96,497 97,495 97,495 96,504 93,508 87,510 83,504 84,507 80,508 79,508 79,508 84,508 87,511 80,514Z", 58, 494],
    ["santa_fe", "Santa Fe", "Pampeana", "santa_fe", "santa_fe", "M193,213 181,228 164,228 180,203 180,199 178,197 178,196 178,193 174,187 175,185 173,180 175,178 179,162 175,158 181,113 221,113 220,116 218,115 218,117 219,122 217,132 212,135 212,137 211,138 210,146 211,149 210,153 209,154 210,162 205,171 204,172 203,175 199,180 196,180 196,183 195,184 196,186 196,191 194,194 196,201 198,205 201,206 201,208 199,214Z", 193, 167],
    ["santiago_del_estero", "Santiago del Estero", "NOA", "santiago_estero", "santiago_del_estero", "M165,146 156,144 156,143 152,143 151,141 149,142 149,141 144,140 137,141 137,138 137,137 138,136 136,135 136,128 132,124 133,124 133,122 132,112 134,111 135,108 134,106 135,105 135,101 136,101 139,92 140,92 140,89 142,89 141,82 145,72 148,73 148,71 156,72 157,71 181,71 181,113 175,158 174,155 170,155 168,154 169,150 166,148Z", 161, 114],
    ["tierra_del_fuego", "Tierra del Fuego", "Patagonia", "tierra_fuego", "tierra_del_fuego", "M144,593 148,593 148,594 151,593 149,594 149,594 148,594 145,595 145,594 144,594 140,596 139,596 139,595 138,595 139,594 141,594 141,593 143,593 143,594ZM113,589 123,592 133,592 129,597 128,596 124,598 124,596 121,596 121,597 114,599 102,596 96,596 88,594 88,595 88,595 84,595 85,596 83,596 83,556 88,561 89,565 88,562 87,563 85,564 85,567 90,568 92,572 96,576 96,577 95,577 96,577 97,576 96,576 97,576 98,577 98,578 100,580 110,585ZM192,541 193,540 193,542 193,542 192,542 193,543 192,544 189,542 190,541 189,541 190,541 191,540 192,541 191,542 190,542ZM201,531 204,532 204,533 204,533 204,533 202,533 202,534 201,533 202,532 203,533 203,533ZM201,536 198,536 198,535 200,535 197,534 196,532 203,535 206,535 204,534 205,533 205,534 209,534 207,534 208,535 211,534 211,534 211,534 212,533 213,534 211,534 214,534 213,533 214,532 217,533 215,534 215,534 217,534 215,536 215,535 214,535 213,536 215,536 213,538 213,537 211,538 212,538 208,541 207,543 205,544 206,544 204,543 204,543 204,544 202,544 202,544 202,544 203,544 203,544 203,544 201,544 201,543 200,543 201,546 202,544 202,544 201,546 200,547 200,546 198,546 200,546 200,547 198,546 197,547 198,547 196,549 195,548 196,548 196,547 195,547 196,547 195,546 193,546 195,547 194,548 190,545 190,544 193,546 194,545 193,545 195,544 193,543 195,543 195,544 195,544 195,544 196,545 196,545 196,545 196,545 196,544 198,545 197,544 200,543 198,543 196,543 198,543 199,542 199,540 200,540 199,541 200,540 200,541 199,542 200,542 201,541 201,540 202,540 203,539 201,539 200,539 200,540 199,540 199,540 198,540 198,541 196,538 198,539 202,538 204,539 205,539 204,539 205,539 204,538ZM208,531 211,531 211,532 212,531 213,532 211,532ZM219,539 219,538 220,538 220,538 217,537 218,535 218,537 219,536 219,535 220,536 221,536 221,536 222,536 221,535 222,536 222,536 222,535 221,535 221,536 221,536 221,536 221,536 217,534 218,534 220,534 219,533 221,533 220,530 223,532 226,533 225,532 226,531 228,532 229,534 228,534 229,534 227,534 226,533 225,535 226,534 226,535 227,535 228,535 227,536 230,536 230,537 230,538 230,537 231,537 231,537 230,536 231,536 228,535 229,535 230,533 234,533 237,535 236,535 235,535 237,536 233,535 233,536 232,535 231,536 237,538 236,538 235,538 235,538 235,538 234,538 236,538 235,538 235,538 238,538 233,540 229,540 230,540 231,540 232,540 230,540 228,541 229,541 230,541 231,541 230,542 230,542 230,542 230,542 230,541 230,541 230,542 230,541 228,542 225,541 226,542 223,542 220,540 220,541 218,541 225,544 224,545 225,545 224,547 222,545 221,545 221,546 220,546 219,544 217,544 217,544 215,544 216,544 215,544 215,544 215,544 217,544 216,546 217,545 218,546 217,546 217,546 218,546 217,547 219,547 218,548 219,549 216,548 216,547 215,547 215,547 213,546 214,547 213,546 213,547 214,547 214,547 213,547 213,547 213,548 214,548 213,548 214,548 215,548 214,549 215,549 214,550 215,550 215,551 214,550 212,550 213,549 212,548 211,549 212,548 210,548 209,546 211,546 210,544 212,544 210,543 212,544 211,543 211,542 212,543 213,543 212,542 213,542 212,541 214,541 214,541 214,542 214,540 215,541 216,539 216,540 219,539 219,540 219,540 218,540 219,540 219,541 219,541 220,540 220,540Z", 91, 578],
    ["tucuman", "Tucumán", "NOA", "tucuman", "tucuman", "M129,110 126,113 125,113 124,109 123,110 121,105 121,101 118,100 122,94 122,93 123,91 120,88 118,89 119,81 124,84 125,79 129,78 131,80 135,83 137,81 142,81 142,89 140,89 140,92 139,92 136,101 135,101 134,111 132,111 132,112Z", 129, 97]]

provincias = pd.DataFrame(PROVINCIAS, columns=["prov_id", "provincia", "region", "col_empleo", "col_expo", "svg_path", "cx", "cy"])

import io
def leer_csv(ruta):   # el servidor rechaza clientes sin User-Agent de navegador
    r = requests.get(f"{CSV_SSPM}/{ruta}", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=120)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), parse_dates=["indice_tiempo"])

filas_prov = []
try:   # Asalariados registrados del sector privado por provincia (miles de personas, mensual)
    emp = leer_csv("154/distribution/154.1/download/asalariados-registrados-sector-privado-segun-provincia-datos-con-estacionalidad.csv")
    emp = emp[emp["indice_tiempo"] >= "2016-01-01"].set_index("indice_tiempo")
    for _, p in provincias.iterrows():
        s_ = emp[f"asalariados_priv_{p.col_empleo}"].dropna()
        filas_prov += [(f, p.prov_id, "emp_priv", v) for f, v in s_.items()]
        filas_prov += [(f, p.prov_id, "emp_priv_ia", v) for f, v in (s_.pct_change(12, fill_method=None) * 100).dropna().items()]
    print("✔ empleo provincial:", emp.index.max().date())
except Exception as e:
    errores.append(("empleo_provincial", str(e)[:150])); print("✘ empleo provincial:", e)

try:   # Exportaciones por provincia y rubro (millones de USD, anual)
    exp_ = leer_csv("350/distribution/350.1/download/exportaciones-provincia-rubro.csv").set_index("indice_tiempo")
    exp_ = exp_[exp_.index >= "2015-01-01"]
    rub = {"pp": "expo_pp", "moa": "expo_moa", "moi": "expo_moi", "cye": "expo_cye"}
    for _, p in provincias.iterrows():
        for r, ind in rub.items():
            c = f"{p.col_expo}_{r}"
            if c in exp_: filas_prov += [(f, p.prov_id, ind, v) for f, v in exp_[c].dropna().items()]
        c = f"{p.col_expo}_total_{p.col_expo}"
        if c in exp_:
            s_ = exp_[c].dropna()
            filas_prov += [(f, p.prov_id, "expo_total", v) for f, v in s_.items()]
            filas_prov += [(f, p.prov_id, "expo_ia", v) for f, v in (s_.pct_change(1, fill_method=None) * 100).dropna().items()]
    print("✔ exportaciones provinciales:", exp_.index.max().date())
except Exception as e:
    errores.append(("exportaciones_provinciales", str(e)[:150])); print("✘ exportaciones provinciales:", e)

prov_datos = pd.DataFrame(filas_prov, columns=["fecha", "prov_id", "indicador", "valor"])
prov_datos = prov_datos.replace([np.inf, -np.inf], np.nan).dropna()
prov_indicadores = pd.DataFrame([
    ("emp_priv",    "Empleo privado registrado",              "miles de personas", "Secretaría de Trabajo (SIPA)", "Mensual", 1),
    ("emp_priv_ia", "Empleo privado registrado - var. % i.a.", "%",                 "Secretaría de Trabajo (SIPA)", "Mensual", 2),
    ("expo_total",  "Exportaciones totales",                  "millones de USD",   "INDEC", "Anual", 3),
    ("expo_ia",     "Exportaciones totales - var. % anual",   "%",                 "INDEC", "Anual", 4),
    ("expo_pp",     "Exportaciones: productos primarios",     "millones de USD",   "INDEC", "Anual", 5),
    ("expo_moa",    "Exportaciones: manufacturas agropecuarias (MOA)", "millones de USD", "INDEC", "Anual", 6),
    ("expo_moi",    "Exportaciones: manufacturas industriales (MOI)",  "millones de USD", "INDEC", "Anual", 7),
    ("expo_cye",    "Exportaciones: combustibles y energía",  "millones de USD",   "INDEC", "Anual", 8),
], columns=["indicador", "nombre", "unidad", "fuente", "frecuencia", "orden"])
print("Datos provinciales:", len(prov_datos), "filas")


✔ empleo provincial: 2026-08-01
✔ exportaciones provinciales: 2024-01-01
Datos provinciales: 7272 filas


## Limpieza y cálculos

In [10]:
# =============================================================
# 3) LIMPIEZA Y CÁLCULOS DERIVADOS
# =============================================================
# Catálogo de indicadores (dimensión para Power BI)
cat_rows = [(ind, nom, blo, uni, fue, tipo, pol, ord_) for (_, ind, nom, blo, uni, fue, tipo, pol, ord_) in SERIES_DATOS]
cat_rows += [(ind, nom, blo, uni, "BCRA", tipo, pol, ord_) for (_, ind, nom, blo, uni, tipo, pol, ord_, _a) in SERIES_BCRA]
cat_rows += [
    ("dolar_mep", "Dólar MEP (bolsa)", "Cambiario y monetario", "$ por USD", "ArgentinaDatos (mercado)", "tipo_cambio", 0, 31.2),
    ("dolar_ccl", "Dólar contado con liquidación", "Cambiario y monetario", "$ por USD", "ArgentinaDatos (mercado)", "tipo_cambio", 0, 31.3),
    ("dolar_blue", "Dólar blue (informal)", "Cambiario y monetario", "$ por USD", "ArgentinaDatos (mercado)", "tipo_cambio", 0, 31.4),
    ("dolar_convertibilidad", "Dólar de convertibilidad (base monetaria / reservas netas)", "Cambiario y monetario", "$ por USD", "BCRA (cálculo propio)", "tipo_cambio", 0, 31.5),
    ("brecha_mep", "Brecha MEP vs. mayorista oficial", "Cambiario y monetario", "%", "BCRA / ArgentinaDatos (cálculo propio)", "tasa_pct", -1, 31.6),
    ("tc_distancia_techo", "Distancia del dólar mayorista al techo de la banda", "Cambiario y monetario", "%", "BCRA (cálculo propio)", "tasa_pct", +1, 33.5),
    ("tamar_real",         "Tasa TAMAR real mensual (vs. inflación)",            "Cambiario y monetario", "% mensual", "BCRA / INDEC (cálculo propio)", "tasa_pct", 0, 35.5),
    ("riesgo_pais",        "Riesgo país (EMBI Argentina)",                       "Externo y fiscal",      "puntos básicos", "ArgentinaDatos (EMBI J.P. Morgan)", "tasa_pct", -1, 47),
]
indicadores = pd.DataFrame(cat_rows, columns=["id_indicador", "nombre", "bloque", "unidad", "fuente", "tipo", "polaridad", "orden"])
TIPO = dict(zip(indicadores.id_indicador, indicadores.tipo))

DESC_REAL = {
    "nominal_pesos": "Valor a pesos constantes del último IPC disponible",
    "saldo_pesos":   "Valor a pesos constantes del último IPC disponible",
    "tipo_cambio":   "Dólar a precios de hoy (deflactado por IPC; aproximación de TC real)",
}
indicadores["valor_real_significa"] = indicadores.tipo.map(DESC_REAL).fillna("")

df = base.copy()
df.loc[df.id_indicador.map(TIPO) == "fraccion", "valor"] *= 100          # 0.079 -> 7.9 %
df = df.drop_duplicates(["fecha", "id_indicador"]).sort_values(["id_indicador", "fecha"])

# IPC para deflactar (se extiende con el último dato si la serie nominal es más nueva)
ipc = df[df.id_indicador == "ipc_general"].set_index("fecha")["valor"]
ipc_ultimo = ipc.iloc[-1]
infl_mensual = ipc.pct_change() * 100

def ipc_en(fechas):
    return ipc.reindex(ipc.index.union(fechas)).ffill().reindex(fechas).values

partes = []
for ind, g in df.groupby("id_indicador"):
    g = g.sort_values("fecha").copy()
    tipo = TIPO.get(ind, "")
    p = 4 if g["frecuencia"].iloc[0] == "quarter" else 12
    v = g["valor"]
    g["var_mensual"] = np.nan; g["var_interanual"] = np.nan
    g["acum_anio_pct"] = np.nan; g["suma_anio"] = np.nan; g["suma_12m"] = np.nan
    g["valor_real"] = np.nan; g["var_real_interanual"] = np.nan

    if tipo in ("tasa_pct", "fraccion", "saldo_usd", "saldo_pesos"):
        g["var_mensual"] = v.diff(1)                    # diferencia (pp o unidades)
        g["var_interanual"] = v.diff(p)
    else:
        g["var_mensual"] = v.pct_change(1, fill_method=None) * 100
        g["var_interanual"] = v.pct_change(p, fill_method=None) * 100

    if tipo == "indice_precios":
        dic_prev = g.set_index("fecha")["valor"]
        base_dic = g["fecha"].apply(lambda f: dic_prev.get(pd.Timestamp(f.year - 1, 12, 1), np.nan))
        g["acum_anio_pct"] = (v / base_dic - 1) * 100

    if tipo in ("flujo_usd", "saldo_usd", "saldo_pesos") or ind == "recaudacion":
        g["suma_anio"] = g.groupby(g["fecha"].dt.year)["valor"].cumsum()
        g["suma_12m"] = v.rolling(12, min_periods=12).sum()

    if tipo in ("nominal_pesos", "saldo_pesos", "tipo_cambio"):
        g["valor_real"] = v * ipc_ultimo / ipc_en(pd.DatetimeIndex(g["fecha"]))
        if tipo != "saldo_pesos":
            g["var_real_interanual"] = g["valor_real"].pct_change(p, fill_method=None) * 100
        else:
            g["suma_12m"] = g["valor_real"].rolling(12, min_periods=12).sum()   # resultado fiscal 12m a pesos de hoy
    partes.append(g)

series = pd.concat(partes, ignore_index=True)

# --- Indicadores derivados ---
piv = series.pivot_table(index="fecha", columns="id_indicador", values="valor")
extra = []
if {"banda_superior", "tc_mayorista"} <= set(piv.columns):
    d = (piv["banda_superior"] / piv["tc_mayorista"] - 1) * 100
    extra += [(f, "tc_distancia_techo", x) for f, x in d.dropna().items()]
if "tamar" in piv.columns:
    tr = ((1 + piv["tamar"] / 1200) / (1 + infl_mensual.reindex(piv.index) / 100) - 1) * 100
    extra += [(f, "tamar_real", x) for f, x in tr.dropna().items()]
if extra:
    e = pd.DataFrame(extra, columns=["fecha", "id_indicador", "valor"]).sort_values(["id_indicador", "fecha"])
    e["frecuencia"] = "month"
    e["var_mensual"] = e.groupby("id_indicador")["valor"].diff(1)
    e["var_interanual"] = e.groupby("id_indicador")["valor"].diff(12)
    series = pd.concat([series, e], ignore_index=True)

series = series[series["fecha"] >= "2017-01-01"].sort_values(["id_indicador", "fecha"]).reset_index(drop=True)
series = series.replace([np.inf, -np.inf], np.nan)
print(series.groupby("id_indicador")["fecha"].agg(["min", "max", "count"]).to_string())


                             min        max  count
id_indicador                                      
badlar                2017-01-01 2026-11-01    119
banda_inferior        2025-04-01 2026-11-01     20
banda_superior        2025-04-01 2026-11-01     20
base_monetaria        2017-01-01 2026-11-01    119
brecha_mep            2018-10-01 2026-11-01     98
desocupacion          2017-01-01 2026-04-01     38
dolar_blue            2017-01-01 2026-11-01    119
dolar_ccl             2017-01-01 2026-11-01    119
dolar_convertibilidad 2017-01-01 2026-11-01    119
dolar_mep             2018-10-01 2026-11-01     98
emae_agro             2017-01-01 2026-08-01    116
emae_construccion     2017-01-01 2026-08-01    116
emae_desest           2017-01-01 2026-08-01    116
emae_industria        2017-01-01 2026-08-01    116
emae_original         2017-01-01 2026-08-01    116
empleo                2017-01-01 2026-04-01     38
exportaciones         2017-01-01 2026-09-01    117
importaciones         2017-01-0

## KPIs, semáforos y resumen automático

In [11]:
# =============================================================
# 4) KPIs (último dato + semáforo) Y RESUMEN AUTOMÁTICO
# =============================================================
MESES = ["enero","febrero","marzo","abril","mayo","junio","julio","agosto","septiembre","octubre","noviembre","diciembre"]
def mes(f):  return f"{MESES[f.month-1]} {f.year}"
def n(x, d=1):
    if x is None or pd.isna(x): return "s/d"
    return f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")

POL = dict(zip(indicadores.id_indicador, indicadores.polaridad))
NOM = dict(zip(indicadores.id_indicador, indicadores.nombre))

def ultimo(ind, k=0):
    g = series[(series.id_indicador == ind) & series.valor.notna()].sort_values("fecha")
    return g.iloc[-1 - k] if len(g) > k else None

def senal(score, tol=0.0):
    if pd.isna(score): return "gris"
    return "verde" if score > tol else ("rojo" if score < -tol else "amarillo")

filas = []
for ind in indicadores.id_indicador:
    g = series[(series.id_indicador == ind) & series.valor.notna()].sort_values("fecha")
    if g.empty: continue
    u = g.iloc[-1]; tipo = TIPO.get(ind, "")
    pol = POL.get(ind, 0)
    # métrica que define el semáforo según el tipo de indicador
    if tipo == "indice_precios":
        prev3 = g["var_mensual"].iloc[-4:-1].mean()
        score = -(u.var_mensual - prev3); tol = 0.2          # verde si la inflación del mes baja vs. promedio 3m
    elif tipo in ("nominal_pesos",) and pd.notna(u.var_real_interanual):
        score = pol * u.var_real_interanual if pol else np.nan; tol = 1.0
    elif tipo in ("saldo_usd", "saldo_pesos"):
        score = u.valor; tol = 0.0
    elif tipo in ("indice_real", "flujo_usd", "stock_usd", "fraccion"):
        score = pol * u.var_interanual if pol else np.nan; tol = 0.5
    elif ind == "inflacion_esperada":
        score = -(u.var_mensual); tol = 0.3
    elif ind == "tc_distancia_techo":
        score = u.valor - 5; tol = 0.0                        # rojo si el dólar queda a menos de 5% del techo
    elif ind == "tamar_real":
        score = u.valor; tol = 0.0
    elif ind == "riesgo_pais":
        score = -u.var_mensual; tol = 25            # verde si baja más de 25 pb en el mes
    else:
        score = np.nan; tol = 0
    filas.append({
        "id_indicador": ind, "fecha": u.fecha, "valor": u.valor,
        "var_mensual": u.var_mensual, "var_interanual": u.var_interanual,
        "var_real_interanual": u.var_real_interanual, "acum_anio_pct": u.acum_anio_pct,
        "suma_12m": u.suma_12m, "valor_real": u.valor_real,
        "tendencia_3m": g["valor"].iloc[-1] - g["valor"].iloc[-4] if len(g) >= 4 else np.nan,
        "senal": senal(score, tol),
    })
kpis = pd.DataFrame(filas)

# ---------- Texto de resumen (se regenera en cada corrida) ----------
def frase_precios():
    u, a = ultimo("ipc_general"), ultimo("ipc_general", 1)
    nu = ultimo("ipc_nucleo"); re = ultimo("ipc_regulados"); ex = ultimo("inflacion_esperada")
    dir_ = "se aceleró" if u.var_mensual > a.var_mensual + 0.05 else ("se desaceleró" if u.var_mensual < a.var_mensual - 0.05 else "se mantuvo estable")
    t = (f"La inflación de {mes(u.fecha)} fue {n(u.var_mensual)}% mensual y {n(u.var_interanual)}% interanual; "
         f"{dir_} frente a {mes(a.fecha)} ({n(a.var_mensual)}%). En el año acumula {n(u.acum_anio_pct)}%. ")
    if nu is not None: t += f"La núcleo marcó {n(nu.var_mensual)}% y los regulados {n(re.var_mensual)}%. "
    if ex is not None: t += f"El mercado (REM) espera {n(ex.valor)}% para los próximos 12 meses."
    return t

def frase_actividad():
    e = ultimo("emae_desest"); eo = ultimo("emae_original"); d = ultimo("desocupacion"); s = ultimo("salarios")
    t = (f"La actividad (EMAE) de {mes(e.fecha)} varió {n(e.var_mensual)}% contra el mes anterior (desestacionalizado) "
         f"y {n(eo.var_interanual)}% interanual. ")
    if d is not None: t += f"La desocupación fue {n(d.valor)}% en el trimestre que empieza en {mes(d.fecha)} ({'+' if d.var_interanual>=0 else ''}{n(d.var_interanual)} pp interanual). "
    if s is not None: t += f"Los salarios ({mes(s.fecha)}) {'ganan' if s.var_real_interanual>=0 else 'pierden'} {n(abs(s.var_real_interanual))}% real interanual."
    return t

def frase_cambiario():
    r = ultimo("reservas"); tc = ultimo("tc_mayorista"); dt_ = ultimo("tc_distancia_techo"); tr = ultimo("tamar_real")
    t = f"Las reservas brutas cerraron {mes(r.fecha)} en USD {n(r.valor,0)} millones ({'+' if r.var_mensual>=0 else ''}{n(r.var_mensual)}% en el mes). "
    t += f"El dólar mayorista promedió ${n(tc.valor,1)} ({n(tc.var_mensual)}% mensual, {n(tc.var_interanual)}% interanual)"
    if dt_ is not None: t += f", a {n(dt_.valor)}% del techo de la banda"
    t += ". "
    if tr is not None: t += f"La tasa TAMAR rinde {n(tr.valor,2)}% mensual en términos reales ({'positiva' if tr.valor>0 else 'negativa'}). "
    mep = ultimo("dolar_mep"); conv = ultimo("dolar_convertibilidad"); br = ultimo("brecha_mep")
    if mep is not None and conv is not None:
        t += (f"El MEP promedió ${n(mep.valor,0)} (brecha de {n(br.valor)}% con el mayorista) y el dólar de convertibilidad "
              f"(base monetaria / reservas netas) se ubica en ${n(conv.valor,0)}: {'por debajo' if conv.valor < tc.valor else 'por encima'} del oficial.")
    return t

def frase_externo_fiscal():
    x = ultimo("saldo_comercial"); ex = ultimo("exportaciones"); im = ultimo("importaciones"); fp = ultimo("resultado_primario"); ff = ultimo("resultado_financiero")
    t = (f"En {mes(x.fecha)} el comercio exterior dejó un saldo de USD {n(x.valor,0)} millones "
         f"(exportaciones {n(ex.var_interanual)}% i.a., importaciones {n(im.var_interanual)}% i.a.); en el año suma USD {n(x.suma_anio,0)} millones. ")
    if fp is not None:
        t += (f"El sector público tuvo en {mes(fp.fecha)} un resultado primario de ${n(fp.valor/1e3,0)} mil millones "
              f"y financiero de ${n(ff.valor/1e3,0)} mil millones ({'superávit' if ff.valor>=0 else 'déficit'}). ")
    rp = ultimo("riesgo_pais")
    if rp is not None:
        t += (f"El riesgo país promedió {n(rp.valor,0)} puntos básicos en {mes(rp.fecha)} "
              f"({'-' if rp.var_mensual < 0 else '+'}{n(abs(rp.var_mensual),0)} pb contra el mes anterior).")
    return t

bloques = [("Precios", frase_precios), ("Actividad y empleo", frase_actividad),
           ("Cambiario y monetario", frase_cambiario), ("Externo y fiscal", frase_externo_fiscal)]
res = []
for b, f in bloques:
    try:   txt = f()
    except Exception as ex: txt = f"(No se pudo generar el texto: {ex})"
    sem = kpis.merge(indicadores[["id_indicador", "bloque"]]).query("bloque == @b")["senal"].value_counts()
    res.append({"bloque": b, "texto": txt, "verdes": int(sem.get("verde", 0)), "amarillos": int(sem.get("amarillo", 0)), "rojos": int(sem.get("rojo", 0))})
resumen = pd.DataFrame(res)
for _, r in resumen.iterrows():
    print(f"■ {r.bloque}: {r.texto}\n")


■ Precios: La inflación de octubre 2026 fue 2,2% mensual y 33,1% interanual; se aceleró frente a septiembre 2026 (1,9%). En el año acumula 26,3%. La núcleo marcó 1,5% y los regulados 2,9%. El mercado (REM) espera 21,8% para los próximos 12 meses.

■ Actividad y empleo: La actividad (EMAE) de agosto 2026 varió 0,1% contra el mes anterior (desestacionalizado) y -1,7% interanual. La desocupación fue 7,9% en el trimestre que empieza en abril 2026 (+0,3 pp interanual). Los salarios (agosto 2026) ganan 1,6% real interanual.

■ Cambiario y monetario: Las reservas brutas cerraron noviembre 2026 en USD 49.600 millones (+2,1% en el mes). El dólar mayorista promedió $1.547,7 (1,5% mensual, 8,4% interanual), a 27,6% del techo de la banda. La tasa TAMAR rinde -0,15% mensual en términos reales (negativa). El MEP promedió $1.583 (brecha de 2,2% con el mayorista) y el dólar de convertibilidad (base monetaria / reservas) se ubica en $980: por debajo del oficial.

■ Externo y fiscal: En septiembre 2026 

## Guardar en Google Sheets

In [12]:
# =============================================================
# 5) GUARDAR EN GOOGLE SHEETS (puente hacia Power BI)
# =============================================================
import gspread
try:
    from google.colab import auth
    auth.authenticate_user()        # en Colab: la primera vez pide permiso a tu cuenta de Google
    from google.auth import default
    creds, _ = default()
except ImportError:                 # corre fuera de Colab (ej. GitHub Actions): usa service account
    import os
    from google.oauth2.service_account import Credentials
    creds = Credentials.from_service_account_file(
        os.environ["GOOGLE_SA_JSON"],
        scopes=["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"])
gc = gspread.authorize(creds)
sh = gc.open_by_key(SPREADSHEET_ID)

def a_hoja(nombre, data):
    d = data.copy()
    for c in d.columns:
        if np.issubdtype(d[c].dtype, np.datetime64):
            d[c] = d[c].dt.strftime("%Y-%m-%d")
        elif np.issubdtype(d[c].dtype, np.floating):
            d[c] = d[c].round(6)
    d = d.astype(object).where(pd.notna(d), "")
    valores = [list(d.columns)] + d.values.tolist()
    try:
        ws = sh.worksheet(nombre)
        ws.clear()
        ws.resize(rows=max(len(valores), 2), cols=max(len(d.columns), 1))
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=nombre, rows=len(valores) + 10, cols=len(d.columns) + 2)
    for i in range(0, len(valores), 20000):          # en tandas para no pasar el límite de la API
        ws.update(values=valores[i:i + 20000], range_name=f"A{i + 1}", value_input_option="RAW")
    print(f"✔ hoja '{nombre}': {len(d)} filas")

a_hoja("series", series[["fecha", "id_indicador", "valor", "var_mensual", "var_interanual", "acum_anio_pct",
                         "suma_anio", "suma_12m", "valor_real", "var_real_interanual", "frecuencia"]])
a_hoja("series_diarias", base_diaria)
a_hoja("indicadores", indicadores)
a_hoja("kpis", kpis)
a_hoja("resumen", resumen)
a_hoja("provincias", provincias.drop(columns=["col_empleo", "col_expo"]))
a_hoja("prov_datos", prov_datos)
a_hoja("prov_indicadores", prov_indicadores)
cobertura = (series.dropna(subset=["valor"]).groupby("id_indicador")["fecha"].agg(desde="min", hasta="max", observaciones="count")
             .reset_index().merge(indicadores[["id_indicador", "nombre", "bloque", "fuente", "orden"]]))
cobertura["frecuencia"] = cobertura["id_indicador"].map(series.groupby("id_indicador")["frecuencia"].first()).map({"month": "Mensual", "quarter": "Trimestral"}).fillna("Mensual")
# alerta de atraso: meses sin dato nuevo vs. el rezago normal de cada serie
hoy_m = pd.Timestamp.today().normalize().replace(day=1)
REZAGO_OK = {"desocupacion": 7, "empleo": 7, "pib_real": 7, "salarios": 5, "salarios_priv_reg": 5,
             "salarios_publico": 5, "ripte": 5, "emae_desest": 4, "emae_original": 4,
             "emae_industria": 4, "emae_construccion": 4, "emae_agro": 4, "inflacion_esperada": 2,
             "emae_pesca": 4, "emae_minas": 4, "emae_electricidad": 4, "emae_comercio": 4, "emae_hoteles": 4, "emae_transporte": 4, "emae_financiero": 4, "emae_inmobiliario": 4, "emae_admin_publica": 4, "emae_ensenianza": 4, "emae_salud": 4, "emae_serv_comunitarios": 4, "emae_impuestos": 4}
cobertura["meses_atraso"] = ((hoy_m.year - cobertura.hasta.dt.year) * 12 + hoy_m.month - cobertura.hasta.dt.month)
cobertura["alerta"] = np.where(cobertura.apply(lambda r: r.meses_atraso > REZAGO_OK.get(r.id_indicador, 3), axis=1), "⚠ fuente atrasada", "ok")
if (cobertura.alerta != "ok").any():
    print("⚠ SERIES ATRASADAS:\n" + cobertura[cobertura.alerta != "ok"][["id_indicador", "hasta", "meses_atraso"]].to_string(index=False))
a_hoja("cobertura", cobertura)
a_hoja("metadata", pd.DataFrame([{
    "ultima_actualizacion": pd.Timestamp.now(tz="America/Argentina/Buenos_Aires").strftime("%Y-%m-%d %H:%M"),
    "fuentes": "INDEC, Secretaría de Hacienda, Secretaría de Trabajo (vía API Series de Tiempo datos.gob.ar), BCRA (API Estadísticas Monetarias v4.0) y ArgentinaDatos (dólares de mercado)",
    "series_con_error": "; ".join(f"{a}: {b}" for a, b in errores) if errores else "ninguna",
}]))

# borrar la hoja vacía por defecto si existe
try:
    sh.del_worksheet(sh.worksheet("Hoja 1"))
except Exception:
    try: sh.del_worksheet(sh.worksheet("Sheet1"))
    except Exception: pass
print("\nListo ✅  →", f"https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}")


✔ hoja 'series': 4184 filas
✔ hoja 'series_diarias': 19610 filas
✔ hoja 'indicadores': 42 filas
✔ hoja 'kpis': 42 filas
✔ hoja 'resumen': 4 filas
✔ hoja 'provincias': 24 filas
✔ hoja 'prov_datos': 7272 filas
✔ hoja 'prov_indicadores': 8 filas
✔ hoja 'cobertura': 42 filas
✔ hoja 'metadata': 1 filas

Listo ✅  → https://docs.google.com/spreadsheets/d/1JkyIFA9AQt4_txuVIHwI9zfNR99iUHNzsujTBAlT_m8


## Deuda pública (Boletín Mensual de Finanzas)

In [ ]:
# =============================================================
# 6) DEUDA PÚBLICA (Boletín Mensual de la Secretaría de Finanzas)
# =============================================================
# El Excel del boletín sale una vez por mes (boletin_mensual_<último día del mes>.xlsx). En cada corrida solo se pregunta si ya
# salió el del mes siguiente al último cargado (una consulta chica); si salió, se arma la serie completa desde el Excel y se
# reescriben las hojas 'deuda' (ancha: fecha + una columna por serie) y 'deuda_series' (catálogo). Si algo falla, las hojas
# quedan como estaban y el resto del ETL no se ve afectado. Código de referencia: scripts/etl/deuda_etl.py
import io, calendar
import openpyxl
import datetime as dt
import re


MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
         "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12}


def _fecha(x):
    if isinstance(x, dt.datetime):
        return dt.date(x.year, x.month, 1)
    if isinstance(x, dt.date):
        return dt.date(x.year, x.month, 1)
    if isinstance(x, str):
        m = re.match(r"\s*([a-záéíóú]{3})[a-z]*[-/ ](\d{2,4})", x.strip().lower())
        if m and m.group(1) in MESES:
            y = int(m.group(2))
            y += 2000 if y < 100 else 0
            return dt.date(y, MESES[m.group(1)], 1)
    return None


def _hoja(wb, nombre, fila_hdr_hint=None):
    """Devuelve (filas, {col: fecha}, lista de bloques de encabezado)."""
    filas = [list(r) for r in wb[nombre].iter_rows(values_only=True)]
    bloques = []
    for i, r in enumerate(filas):
        cols = {j: _fecha(c) for j, c in enumerate(r) if _fecha(c)}
        if len(cols) >= 20:
            bloques.append((i, cols))
    return filas, bloques


def _etiqueta(r):
    for x in r[:3]:
        if isinstance(x, str) and x.strip():
            return x.strip()
    return ""


def _etiquetas(r):
    return [x.strip() for x in r[:3] if isinstance(x, str) and x.strip()]


def _fila(filas, idx, cols):
    r = filas[idx]
    out = {}
    for j, f in cols.items():
        v = r[j] if j < len(r) else None
        if isinstance(v, (int, float)):
            out[f] = float(v)
    return out


def _suma(*series):
    fechas = set().union(*[set(s) for s in series])
    return {f: sum(s.get(f, 0.0) for s in series) for f in fechas}


def _resta(a, *bs):
    return {f: a[f] - sum(b.get(f, 0.0) for b in bs) for f in a}


def construir(ruta_xlsx):
    wb = openpyxl.load_workbook(ruta_xlsx, data_only=True, read_only=True)
    filas = []

    def agregar(id_, grupo, nombre, orden, serie):
        for f, v in sorted(serie.items()):
            filas.append({"fecha": f.isoformat(), "id": id_, "grupo": grupo, "serie": nombre,
                          "valor": round(v, 3), "orden": orden})

    # ---- A.1: total e instrumentos -------------------------------------------------------
    f1, b1 = _hoja(wb, "A.1")
    c1 = b1[0][1]

    def fila_por_etiqueta(filas_, cols, etiqueta, desde=0, hasta=None, exacta=True):
        for i in range(desde, hasta or len(filas_)):
            es = _etiquetas(filas_[i])
            if any((e == etiqueta) if exacta else e.startswith(etiqueta) for e in es):
                return _fila(filas_, i, cols)
        raise KeyError(etiqueta)

    total = fila_por_etiqueta(f1, c1, "A- DEUDA BRUTA ( I + II  + III)", exacta=False)
    normal = fila_por_etiqueta(f1, c1, "I- DEUDA EN SITUACIÓN DE PAGO NORMAL")
    titulos = fila_por_etiqueta(f1, c1, "TÍTULOS PÚBLICOS")
    # Letras del Tesoro: mediano y largo plazo + corto plazo
    idx_letras = [i for i, r in enumerate(f1) if _etiqueta(r) == "LETRAS DEL TESORO"]
    letras = _suma(*[_fila(f1, i, c1) for i in idx_letras[:2]])
    org_int = fila_por_etiqueta(f1, c1, "ORGANISMOS INTERNACIONALES")
    fmi = fila_por_etiqueta(f1, c1, "FMI")
    org_sin_fmi = _resta(org_int, fmi)
    otros = _resta(normal, titulos, letras, org_int)

    g = "Total"
    agregar("deuda_total", g, "Deuda bruta total", 1, total)
    agregar("deuda_pago_normal", g, "Deuda en situación de pago normal", 2, normal)
    g = "Instrumento o acreedor"
    agregar("inst_titulos", g, "Títulos públicos", 1, titulos)
    agregar("inst_letras", g, "Letras del Tesoro", 2, letras)
    agregar("inst_org_sin_fmi", g, "Organismos internacionales (sin FMI)", 3, org_sin_fmi)
    agregar("inst_fmi", g, "FMI", 4, fmi)
    agregar("inst_otros", g, "Otros préstamos, adelantos BCRA y avales", 5, otros)

    # ---- A.2: legislación -------------------------------------------------------------------
    f2, b2 = _hoja(wb, "A.2")
    c2 = b2[0][1]
    leg_arg = fila_por_etiqueta(f2, c2, "I- LEGISLACIÓN ARGENTINA")
    leg_ext = fila_por_etiqueta(f2, c2, "II- LEGISLACIÓN EXTRANJERA")
    g = "Legislación"
    agregar("leg_arg", g, "Legislación argentina (interna)", 1, leg_arg)
    agregar("leg_ext", g, "Legislación extranjera (externa)", 2, leg_ext)

    # ---- A.3: moneda y tasa (deuda en situación de pago normal) ------------------------------
    f3, b3 = _hoja(wb, "A.3")
    c3 = b3[0][1]
    ini = b3[0][0]

    def bloque(desde_etq, hasta_etq):
        i0 = next(i for i in range(ini, len(f3)) if _etiqueta(f3[i]).startswith(desde_etq))
        i1 = next(i for i in range(i0 + 1, len(f3)) if _etiqueta(f3[i]).startswith(hasta_etq))
        return i0, i1

    i_loc, i_ext = bloque("Moneda local", "Moneda extranjera")
    i_fin = next(i for i in range(i_ext + 1, len(f3)) if _etiqueta(f3[i]).startswith("COMPOSICIÓN POR MONEDA"))

    def tasas(i0, i1):
        """Dentro de [i0, i1) suma las filas 'Tasa ...' por tipo (fija, cero, variable)."""
        res = {"fija": {}, "cero": {}, "variable": {}}
        for i in range(i0, i1):
            e = _etiqueta(f3[i]).lower()
            for k in res:
                if e == "tasa " + k:
                    res[k] = _suma(res[k], _fila(f3, i, c3)) if res[k] else _fila(f3, i, c3)
        return res

    loc = tasas(i_loc, i_ext)
    ext = tasas(i_ext, i_fin)
    loc_con = _suma(loc["fija"], loc["variable"])
    loc_sin = loc["cero"]
    ext_con = _suma(ext["fija"], ext["variable"])
    ext_sin = ext["cero"]
    g = "Moneda y tasa"
    agregar("mt_loc_con", g, "Moneda local, con intereses (tasa fija o variable)", 1, loc_con)
    agregar("mt_loc_sin", g, "Moneda local, sin intereses (tasa cero)", 2, loc_sin)
    agregar("mt_ext_con", g, "Moneda extranjera, con intereses (tasa fija o variable)", 3, ext_con)
    agregar("mt_ext_sin", g, "Moneda extranjera, sin intereses (tasa cero)", 4, ext_sin)

    # control: las cuatro partes deben sumar la deuda en pago normal
    for f, v in normal.items():
        s = loc_con.get(f, 0) + loc_sin.get(f, 0) + ext_con.get(f, 0) + ext_sin.get(f, 0)
        assert abs(s - v) < 5, ("A.3 no cierra", f, s, v)

    # ---- A.5: pagos de capital e intereses por moneda de pago -----------------------------
    f5, b5 = _hoja(wb, "A.5")
    c5 = b5[0][1]
    i_mn = next(i for i, r in enumerate(f5) if _etiqueta(r).startswith("- MONEDA NACIONAL"))
    i_me = next(i for i, r in enumerate(f5) if _etiqueta(r).startswith("- MONEDA EXTRANJERA"))

    def pago(i0, nombre):
        for i in range(i0 + 1, i0 + 6):
            if _etiqueta(f5[i]) == nombre:
                return _fila(f5, i, c5)
        raise KeyError(nombre)

    g = "Pagos del mes"
    agregar("pago_mn_capital", g, "Capital en moneda nacional", 1, pago(i_mn, "CAPITAL"))
    agregar("pago_mn_interes", g, "Intereses en moneda nacional", 2, pago(i_mn, "INTERES"))
    agregar("pago_me_capital", g, "Capital en moneda extranjera", 3, pago(i_me, "CAPITAL"))
    agregar("pago_me_interes", g, "Intereses en moneda extranjera", 4, pago(i_me, "INTERES"))

    return filas


HEADERS_WEB = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}


def _url_boletin(anio, mes):
    return ("https://www.argentina.gob.ar/sites/default/files/boletin_mensual_"
            f"{calendar.monthrange(anio, mes)[1]:02d}_{mes:02d}_{anio}.xlsx")


def _a_fecha(v):
    """Fecha de la hoja: número de serie de Sheets, texto ISO o texto d/m/a."""
    if isinstance(v, (int, float)):
        return (dt.date(1899, 12, 30) + dt.timedelta(days=int(v))).replace(day=1)
    s = str(v).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            d = dt.datetime.strptime(s[:10], fmt).date()
            return d.replace(day=1)
        except ValueError:
            pass
    return None


def _meses_candidatos(ultimo):
    """Meses a probar, del más nuevo al más viejo: desde el mes siguiente al último cargado hasta el mes actual."""
    hoy = dt.date.today()
    y, m = (ultimo.year, ultimo.month) if ultimo else (hoy.year - 1, hoy.month)
    out = []
    while True:
        m += 1
        if m == 13: y, m = y + 1, 1
        if (y, m) > (hoy.year, hoy.month): break
        out.append((y, m))
    return out[::-1][:6]


try:
    try:
        ws_deuda = sh.worksheet("deuda")
        col_f = [v for v in ws_deuda.col_values(1, value_render_option="UNFORMATTED_VALUE")[1:] if v != ""]
        ultimo_cargado = max(f for f in (_a_fecha(v) for v in col_f) if f)
    except (gspread.WorksheetNotFound, ValueError):
        ultimo_cargado = None
    crudo = None
    for (y_, m_) in _meses_candidatos(ultimo_cargado):
        r_ = requests.get(_url_boletin(y_, m_), headers=HEADERS_WEB, timeout=90)
        if r_.status_code == 200 and r_.content[:2] == b"PK":
            crudo, mes_boletin = r_.content, (y_, m_)
            break
    if crudo is None:
        print("deuda: todavía no hay un boletín más nuevo que", ultimo_cargado.isoformat() if ultimo_cargado else "(sin datos)")
    else:
        largo = pd.DataFrame(construir(io.BytesIO(crudo)))
        orden_series = largo.drop_duplicates("id")[["id", "grupo", "serie", "orden"]]
        ancho = largo.pivot(index="fecha", columns="id", values="valor")[list(orden_series["id"])].reset_index()
        ancho["fecha"] = pd.to_datetime(ancho["fecha"])
        catalogo = orden_series.rename(columns={"orden": "orden_grupo"})
        assert len(ancho) >= 60 and ancho["fecha"].max() >= pd.Timestamp(year=mes_boletin[0], month=mes_boletin[1], day=1) - pd.DateOffset(months=1)
        a_hoja("deuda", ancho)
        a_hoja("deuda_series", catalogo)
        print(f"✔ deuda: boletín de {mes_boletin[1]:02d}/{mes_boletin[0]} ({len(ancho)} meses, {len(catalogo)} series)")
except Exception as e_deuda:
    print("⚠ deuda: no se actualizó (", repr(e_deuda)[:300], ") - las hojas quedan como estaban")
